# Notebook 04 â€” End-to-End Demo (Gradio)

**Goal:** Combine the two trained models into an interactive demo:
1. Upload Indonesian audio
2. Whisper (fine-tuned) â†’ transcript
3. IndoBERT (fine-tuned) â†’ content category + confidence

**Prerequisites:** Run Notebooks 01, 02, 03 first (or have checkpoints on Drive).

Gradio will generate a public share link â€” paste it in your README/portfolio.

## 1. Setup

In [ ]:
!pip install -q transformers gradio librosa soundfile torch accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/indonesian-audio-intelligence'
if not os.path.exists(f'{REPO_DIR}/src'):
    !git clone https://github.com/KimiDandy/indonesian-audio-intelligence {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR}')

import sys
sys.path.append(REPO_DIR)

WHISPER_CKPT  = f'{REPO_DIR}/whisper-id-finetuned'
INDOBERT_CKPT = f'{REPO_DIR}/checkpoints/indobert-best'

for path in [WHISPER_CKPT, INDOBERT_CKPT]:
    if os.path.exists(path):
        print(f'Found: {path}')
    else:
        print(f'MISSING: {path} — run the corresponding notebook first!')

## 2. Load Both Models

In [ ]:
import torch
import numpy as np
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading Whisper...')
whisper_processor = WhisperProcessor.from_pretrained(WHISPER_CKPT)
whisper_model     = WhisperForConditionalGeneration.from_pretrained(WHISPER_CKPT).to(device)
whisper_model.eval()

# Force Indonesian
whisper_model.config.forced_decoder_ids = whisper_processor.get_decoder_prompt_ids(
    language='indonesian', task='transcribe'
)
print('Whisper loaded.')

In [ ]:
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print('Loading IndoBERT...')
bert_tokenizer = AutoTokenizer.from_pretrained(INDOBERT_CKPT)
bert_model     = AutoModelForSequenceClassification.from_pretrained(INDOBERT_CKPT).to(device)
bert_model.eval()

# Load label mapping from classification results
with open(f'{REPO_DIR}/classification_results.json') as f:
    clf_results = json.load(f)
CATEGORIES   = clf_results['categories']
IDX_TO_LABEL = {int(v): k for k, v in CATEGORIES.items()}

print(f'IndoBERT loaded. Labels: {list(CATEGORIES.keys())}')

## 3. Inference Pipeline

In [ ]:
SAMPLE_RATE = 16_000

def transcribe(audio_path: str) -> str:
    """Load audio file â†’ Whisper â†’ Indonesian transcript."""
    audio_array, sr = librosa.load(audio_path, sr=None)
    if sr != SAMPLE_RATE:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLE_RATE)

    inputs = whisper_processor(
        audio_array, sampling_rate=SAMPLE_RATE, return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        predicted_ids = whisper_model.generate(**inputs)

    return whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

In [ ]:
import torch.nn.functional as F

def classify(text: str) -> tuple[str, float, dict]:
    """Indonesian text â†’ category label + confidence scores."""
    inputs = bert_tokenizer(
        text, truncation=True, max_length=128,
        return_tensors='pt', padding=True
    ).to(device)

    with torch.no_grad():
        logits = bert_model(**inputs).logits

    probs = F.softmax(logits, dim=-1)[0].cpu()
    pred_idx   = probs.argmax().item()
    pred_label = IDX_TO_LABEL[pred_idx]
    confidence = probs[pred_idx].item()

    # All class scores for Gradio Label component
    all_scores = {IDX_TO_LABEL[i]: float(probs[i]) for i in range(len(IDX_TO_LABEL))}
    return pred_label, confidence, all_scores

In [ ]:
def predict_audio(audio_file):
    """Full pipeline: audio path â†’ transcript + category + confidence."""
    if audio_file is None:
        return 'No audio uploaded.', '', ''

    try:
        transcript = transcribe(audio_file)
    except Exception as e:
        return f'Transcription error: {e}', '', ''

    if not transcript.strip():
        return '(no speech detected)', 'unknown', '0%'

    label, confidence, _ = classify(transcript)
    return transcript, label.upper(), f'{confidence:.1%}'

# Quick test (if you have a local .wav file on Colab)
# transcript, label, conf = predict_audio('/path/to/test.wav')
# print(transcript, label, conf)

## 4. Gradio Interface

In [ ]:
import gradio as gr

with gr.Blocks(title='Indonesian Audio Content Classifier') as demo:
    gr.Markdown("""
    # Indonesian Audio Content Classifier
    Upload audio in **Bahasa Indonesia** â†’ get transcript + content category.

    **Pipeline:** Whisper-small (fine-tuned on Common Voice ID) â†’ IndoBERT (fine-tuned on id_liputan6)
    """)

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(
                sources=['upload', 'microphone'],
                type='filepath',
                label='Input Audio (Indonesian)',
            )
            submit_btn = gr.Button('Analyze', variant='primary')

        with gr.Column():
            transcript_out = gr.Textbox(label='Transcript', lines=4, interactive=False)
            category_out   = gr.Textbox(label='Category', interactive=False)
            confidence_out = gr.Textbox(label='Confidence', interactive=False)

    gr.Markdown("""
    **Categories:** Bisnis Â· Teknologi Â· Otomotif Â· Bola Â· Health Â· Lifestyle Â· Showbiz Â· Regional

    > Model source: [indonesian-audio-intelligence](https://github.com/your-username/indonesian-audio-intelligence)
    """)

    submit_btn.click(
        fn=predict_audio,
        inputs=[audio_input],
        outputs=[transcript_out, category_out, confidence_out],
    )

demo.launch(share=True, debug=False)

## 5. Next Steps

- Copy the generated `share.gradio.live` URL into your README
- Take a screenshot of the running demo for portfolio visuals
- (Optional) Deploy permanently to HuggingFace Spaces: `gradio deploy`